# Variance Reduction

Variance reduction techniques improve Monte Carlo simulations by reducing estimator variance without increasing the number of simulated paths.

This notebook demonstrates variance reduction for Asian options using the **control variates method**:

- Compute the price of a Asian option based on **geometric averages**
- Estimate the geometric Asian option price via **Monte Carlo simulation**
- Apply **control variates** to price an arithmetic Asian option
- Evaluate the efficiency and accuracy of the method under different parameter settings (number of paths, strike price, monitoring dates)

## 1. Analytical Solution for an Asian Call Option with Geometric Averages

Under the Black–Scholes model, the stock price follows the risk–neutral SDE

$$
dS(t) = rS(t)dt + \sigma S(t)dW(t)
$$

whose solution is

$$
S(T) = S(t_0)\exp\left((r-\tfrac{1}{2}\sigma^2)T + \sigma \sqrt{T}Z \right),
$$

where  

- $r$ = risk-free rate  
- $\sigma$ = volatility  
- $Z \sim N(0,1)$  

Hence $S(T)$ is **lognormally distributed**. Assume the stock price is observed $N$ times before maturity

$$
t_i = \frac{iT}{N}, \quad i=0,\dots,N
$$

The **geometric average price** is

$$
\tilde{A}_N =
\left(\prod_{i=0}^{N} S(t_i)\right)^{\frac{1}{N+1}}
$$

The payoff of a geometric Asian call is

$$
(\tilde{A}_N - K)^+
$$

Using the ratio representation of prices,

$$
\prod_{i=1}^{N} S(t_i)
=
\frac{S(t_N)}{S(t_{N-1})}
\left(\frac{S(t_{N-1})}{S(t_{N-2})}\right)^2
\cdots
\left(\frac{S(t_1)}{S(t_0)}\right)^N S(t_0)^{N}
$$

From the GBM solution,

$$
\frac{S(t_i)}{S(t_{i-1})}
=
\exp\left(
(r-\tfrac{1}{2}\sigma^2)\frac{T}{N}
+
\sigma\sqrt{\frac{T}{N}}X_i
\right)
$$

where

$$
X_i \sim N(0,1)
$$

Taking the logarithm of the normalized geometric average

$$
\log\left(
\frac{\tilde{A}_N}{S(t_0)}
\right)
=
\frac{1}{N+1}
\sum_{i=1}^{N}
i
\left[
(r-\tfrac{1}{2}\sigma^2)\frac{T}{N}
+
\sigma\sqrt{\frac{T}{N}}X_i
\right]
$$

and using

$$
1 + 2 + \cdots + N = \frac{N(N+1)}{2}
$$

we obtain

$$
\log\left(\frac{\tilde{A}_N}{S(t_0)}\right)
=
(r-\tfrac{1}{2}\sigma^2)\frac{T}{2}
+
\sigma\sqrt{\frac{T}{N}}
\frac{\sum_{i=1}^N iX_i}{N+1}
$$

Since $X_i$ are i.i.d. $N(0,1)$,

$$
\frac{\sigma\sqrt{T/N}}{N+1}\sum_{i=1}^{N} iX_i
\sim
N\left(
0,
\frac{(2N+1)\sigma^2T}{6(N+1)}
\right)
$$

By defining the effective volatility as

$$
\tilde{\sigma}
=
\sigma
\sqrt{\frac{2N+1}{6(N+1)}}
$$

we get

$$
\log\left(\frac{\tilde{A}_N}{S(t_0)}\right)
=
(\tilde{r}-\tfrac12\tilde{\sigma}^2)T
+
\tilde{\sigma}\sqrt{T}Z
$$

where

$$
\tilde{r}
=
\left(r-\tfrac12\sigma^2\right)
+
\frac{\tilde{\sigma}^2}{2}
$$

So, the geometric Asian option price becomes ($t_0=0$)

$$
C_g^A(S_0,T)
=
e^{-rT}\mathbb{E}\left[(\tilde{A}_N-K)^+\right]
$$

which can be written as

$$
C_g^A(S_0,T)
=
e^{(\tilde{r}-r)T}C_{BS}(S_0,T;\tilde{r},\tilde{\sigma})
$$

where $C_{BS}$ is the Black–Scholes call price.

In [ ]:
import numpy as np
from scipy.stats import norm

# Parameters
S0 = 100  # Initial stock price
T = 1     # Time to maturity (1 year)
N = 50    # Number of time intervals for averaging
K = 99    # Strike price
r = 0.06  # Risk-free interest rate
sigma = 0.2 # Volatility
M = 10000   # Number of Monte Carlo simulation paths 

In [9]:
def geo_asian_analytic(S0, K, T, r, sigma, N):
  
    # Adjusted volatility
    sigma_tilde = sigma * np.sqrt((2*N + 1) / (6*(N + 1)))
    
    # Adjusted drift
    r_tilde = ((r - 0.5 * sigma**2) + sigma_tilde**2) / 2
    
    # d1 and d2
    d1 = (np.log(S0/K) + (r_tilde + 0.5 * sigma_tilde**2) * T) / (sigma_tilde * np.sqrt(T))
    d2 = (np.log(S0/K) + (r_tilde - 0.5 * sigma_tilde**2) * T) / (sigma_tilde * np.sqrt(T))
    
    # Analytical price
    call_price = np.exp(-r*T) * (S0 * np.exp(r_tilde*T) * norm.cdf(d1) - K * norm.cdf(d2))
    
    return call_price

print("Analytical Geometric Asian Call Price:", geo_asian_analytic(S0, K, T, r, sigma, N))

Analytical Geometric Asian Call Price: 6.310747867416329


## 2. Monte Carlo Simulation for the Asian Call Option

We simulate stock paths under GBM 
$$
S_{t+\Delta t} = S_t \exp\left(\left(r - \frac{1}{2}\sigma^2 \right)\Delta t + \sigma \sqrt{\Delta_t}Z\right)
$$

In [4]:
dt = T / N

payoffs = np.zeros(M)

for m in range(M):

    S = S0
    prices = []

    for i in range(N):
        Z = np.random.normal()
        S = S * np.exp((r - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z)
        prices.append(S)

    # geometric average
    geo_avg = np.exp(np.mean(np.log(prices)))

    payoffs[m] = max(geo_avg - K, 0)

# Monte Carlo price
C_geo_MC = np.exp(-r*T) * np.mean(payoffs)

print("Monte Carlo Geometric Asian Call Price:", C_geo_MC)

std_error = np.exp(-r*T) * np.std(payoffs) / np.sqrt(M)

print("Standard Error:", std_error)

Monte Carlo Geometric Asian Call Price: 6.4082459307978565
Standard Error: 0.08142515899424842
